Pandas 常以 NaN、NaT 表示缺失值。

缺失不一定代表「資料錯了」，它可能來自很多情況，例如：

- 使用者忘記填寫
- 系統沒有成功取得資料
- 該欄位不適用
- 使用者拒絕提供
- 資料匯入時格式錯誤

所以看到缺失值時，不是立刻全部刪掉，而是先判斷這個欄位對分析有多重要。

```py
df.isna() # 檢查每個位置是否為缺失值
df.isna().sum() # 統計每個欄位有多少個缺失值
```

### 策略一：刪除缺失資料

有些欄位不能缺。

例如「交易編號」如果不存在，後續就很難辨認這筆交易，也可能無法和其他資料表進行 merge。這種情況可以直接刪除。

In [2]:
import pandas as pd


### 建立範例資料
data = {
    "交易編號": ["T001", "T002", None, "T004"],
    "類別": ["餐飲", "交通", "購物", "餐飲"],
    "金額": [120, 250, 300, None]
}

df = pd.DataFrame(data)

### dropna() 用來刪除含有缺失值的資料列。
df = df.dropna(
    subset=["交易編號"] # 指定「我要檢查哪些欄位」
) 

df

,交易編號,類別,金額
0,T001,餐飲,120.0
1,T002,交通,250.0
3,T004,餐飲,NaN


### 策略二：填補缺失值

有些資料雖然缺失，但不需要直接刪掉。

In [4]:
import pandas as pd


### 建立範例資料
data = {
    "交易編號": ["T001", "T002", "T003", "T004"],
    "類別": ["餐飲", None, "交通", None],
    "金額": [120, 250, None, 180]
}

df = pd.DataFrame(data)

### 填補缺失值
df["類別"] = df["類別"].fillna("未分類") # 將「類別」缺失的位置填成「未分類」
df

,交易編號,類別,金額
0,T001,餐飲,120.0
1,T002,未分類,250.0
2,T003,交通,NaN
3,T004,未分類,180.0


In [5]:
# 數值資料也可以填補。
# 例如有一筆「金額」缺失，就用金額中位數去補
median_amount = df["金額"].median() # 計算金額中位數

df["金額"] = df["金額"].fillna(median_amount) # 使用中位數填補缺失金額
df

,交易編號,類別,金額
0,T001,餐飲,120.0
1,T002,未分類,250.0
2,T003,交通,180.0
3,T004,未分類,180.0


### 策略三：保留缺失值，另外建立標記

有時候「缺失」本身就是資訊。

例如「金額沒有填」，可能代表：

- 系統取得資料失敗
- 使用者沒有提供
- 某種特殊交易沒有金額
- 資料串接發生問題

如果直接把缺失值填掉，這些資訊就消失了。這時可以建立一個新的欄位，記錄原本是否缺失。

In [6]:
import pandas as pd


### 建立範例資料
data = {
    "交易編號": ["T001", "T002", "T003", "T004"],
    "類別": ["餐飲", "交通", "購物", "餐飲"],
    "金額": [120, None, 300, None]
}

df = pd.DataFrame(data)

### 建立缺失標記欄位
df["金額是否缺失"] = df["金額"].isna()
df

,交易編號,類別,金額,金額是否缺失
0,T001,餐飲,120.0,False
1,T002,交通,NaN,True
2,T003,購物,300.0,False
3,T004,餐飲,NaN,True


## Problem. 缺失值實作練習

In [7]:
import pandas as pd

### 建立練習資料
data = {
    "交易編號": ["T001", "T002", None, "T004", "T005", "T006"],
    "類別": ["餐飲", None, "交通", "購物", None, "餐飲"],
    "品項": ["珍珠奶茶", "捷運", "公車", None, "耳機", "雞排"],
    "金額": [80, 50, 30, 1200, None, None]
}

df = pd.DataFrame(data)
df

,交易編號,類別,品項,金額
0,T001,餐飲,珍珠奶茶,80.0
1,T002,NaN,捷運,50.0
2,NaN,交通,公車,30.0
3,T004,購物,NaN,1200.0
4,T005,NaN,耳機,NaN
5,T006,餐飲,雞排,NaN


### Problem. 找出資料缺失狀況
請使用 Pandas 檢查
- 每一個資料位置是否為缺失值。
- 每個欄位分別有幾個缺失值。

### Problem. 刪除沒有交易編號的資料

「交易編號」是每筆交易的重要識別資料。

如果交易編號缺失，這筆資料無法確認是哪一筆交易，因此決定將整列刪除。

請完成：
- 只檢查「交易編號」欄位。
- 如果「交易編號」缺失，刪除該筆資料。
- 將處理結果重新存回 df。

### Problem. 填補缺失的交易類別

資料中有些交易沒有填寫「類別」，但我們不希望因此刪除整筆交易。

請將「類別」中的缺失值全部填成「未分類」

處理完成後顯示 DataFrame，確認原本的 NaN 已經變成「未分類」

### Problem. 使用中位數填補金額

「金額」欄位有部分資料缺失。

直接填 0 可能讓後續平均金額受到影響，因此這次使用現有資料的中位數填補。

請完成：

- 計算「金額」欄位的中位數。
- 將結果存入 median_amount。
- 使用 median_amount 填補「金額」的缺失值。
- 顯示處理完成的資料。

### Problem. 不要急著補，記錄哪些品項原本缺失

「品項沒有填可能代表資料匯入出了問題，我不希望你直接把缺失值蓋掉，我想知道原本哪些資料有問題。」因此不要刪除，也不要填補「品項」。

請新增一個欄位「品項是否缺失」，如果「品項」原本是缺失值，結果為 True，否則為 False。

